# 🦙 Ollama + Open WebUI on Google Colab

Run a local LLM (GGUF model) on a Colab T4 GPU with a full Open WebUI frontend, exposed via Pinggy tunnels.

**Stack:** Ollama · Open WebUI · Pinggy · Gemma-4 Uncensored (Q8 GGUF)

---
### Steps
1. Install all dependencies
2. Start Ollama & verify
3. Download the model
4. Register model with Ollama
5. Expose tunnels (Ollama API + WebUI)
6. Launch Open WebUI


---
**Make sure every cell is complete one by one.
It's might take time before it complete.
wait for the green tick before continue to next cell.**

In [1]:
#@title 🎨 Design Elements Download
#@markdown > Essential For Next Steps
# ── Helpers ──────────────────────────────────────────────────────────────────
from IPython.display import display, HTML

def success(msg):
    display(HTML(f'<p style="color:#2ecc71;font-weight:bold;font-size:15px">&#x2705; {msg}</p>'))

def failure(msg):
    display(HTML(f'<p style="color:#e74c3c;font-weight:bold;font-size:15px">&#x274C; {msg}</p>'))

def info(msg):
    display(HTML(f'<p style="color:#3498db;font-weight:bold;font-size:15px">&#x1F517; {msg}</p>'))

In [ ]:
#@title 📦 Download Dependencies
# ── Step 1: Install system tools, Ollama, and all Python packages ────────────
import subprocess

steps = [
    ("System packages (lshw, pciutils, zstd)", "apt-get install -y -q lshw pciutils zstd"),
    ("Ollama",                                 "curl -fsSL https://ollama.com/install.sh | sh"),
    ("Python packages",                        "pip install -q requests pinggy open-webui"),
]

all_ok = True
for label, cmd in steps:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        success(f"{label} installed.")
    else:
        failure(f"{label} failed: {result.stderr.strip()[:200]}")
        all_ok = False

if all_ok:
    success("All dependencies ready — proceed to Step 2.")

In [ ]:
#@title 🚀 Start Ollama Server
# ── Step 2: Start Ollama server and verify it's listening ────────────────────
import subprocess
import time

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

try:
    output = subprocess.run(
        ["sudo", "lsof", "-i", "-P", "-n"],
        capture_output=True, text=True
    ).stdout
    if ":11434 (LISTEN)" in output:
        success("Ollama is listening on port 11434.")
    else:
        failure("Ollama not detected on port 11434 — try re-running this cell.")
except Exception as e:
    failure(f"Port check error: {e}")

In [ ]:
#@title 📥 Download Model
#@markdown > Might take some time because it's a 7.5 GB Model
# ── Step 3: Download the GGUF model from HuggingFace (~7.5 GB) ───────────────
import subprocess
import os

MODEL_URL = (
    "https://huggingface.co/HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive"
    "/resolve/main/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf"
)
MODEL_PATH = "/content/model.gguf"

if os.path.exists(MODEL_PATH):
    success("Model already downloaded — skipping.")
else:
    result = subprocess.run(
        ["wget", "-q", "-O", MODEL_PATH, MODEL_URL],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        success("Model downloaded successfully.")
    else:
        failure(f"Download failed: {result.stderr.strip()[:200]}")

In [ ]:
#@title 🔧 Register Model with Ollama
# ── Step 4: Register the GGUF model with Ollama ──────────────────────────────
import subprocess

MODEL_NAME = "gemma4-uncensored"

with open("Modelfile", "w") as f:
    f.write("FROM /content/model.gguf\n")

result = subprocess.run(
    ["ollama", "create", MODEL_NAME, "-f", "Modelfile"],
    capture_output=True, text=True
)
if result.returncode == 0:
    success(f"Model '{MODEL_NAME}' registered with Ollama.")
else:
    failure(f"Model registration failed: {result.stderr.strip()[:200]}")

In [ ]:
#@title 🔗 Open Tunnels
# ── Step 5: Open Pinggy tunnels (Ollama API + Open WebUI) ────────────────────
import pinggy

try:
    tunnel_ollama = pinggy.start_tunnel(
        forwardto="localhost:11434",
        headermodification=["u:Host:localhost:11434"]
    )
    info(f"Ollama API  →  {tunnel_ollama.urls[1]}")
    success("Ollama tunnel is live.")
except Exception as e:
    failure(f"Ollama tunnel failed: {e}")

try:
    tunnel_webui = pinggy.start_tunnel(forwardto="localhost:8000")
    info(f"Open WebUI  →  {tunnel_webui.urls[1]}")
    success("WebUI tunnel is live — copy the link above before running Step 6.")
except Exception as e:
    failure(f"WebUI tunnel failed: {e}")

#Caution ⚠️
---
It's need some time to load the Server so wait 1-2 minutes before opening the url [Open WebUI]

The first message of the model takes time so don't stop the response in Open WebUI chat.

⛔ Don't Close the tab and stop running this cell..



In [ ]:
#@title 🌐 Launch Open WebUI
#@markdown > The First Response From the Model Takes 4-5 minutes because of the model loading on GPU. After that response will be normal time.
# ── Step 6: Launch Open WebUI (open your tunnel URL in the browser) ──────────
import subprocess
import time

proc = subprocess.Popen(
    ["open-webui", "serve", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(8)

if proc.poll() is None:
    success("Open WebUI is running — open your tunnel URL now.")
else:
    failure("Open WebUI failed to start — check your installation.")

proc.wait()  # Keep the session alive